In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch
import csv
from sklearn import datasets
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
import numpy as np

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("microsoft/codebert-base",force_download=False)
model = AutoModel.from_pretrained("microsoft/codebert-base",force_download=False)

In [ ]:
#find . -name \*.java -exec cp {} ..\newDir \;

#pos classes
entity=[]
application=[]
utility=[]
data={}
inp = open("data/posapplication.txt", "r")
application = inp.read().splitlines()
inp.close()
for c in application:
    z = c.split(".")
    elem=z[-1].strip().lower()
    data[elem]=["application"]
inp = open("data/posutility.txt", "r")
utility = inp.read().splitlines()
inp.close()
for c in utility:
    z = c.split(".")
    elem=z[-1].strip().lower()
    data[elem]=["utility"]
inp = open("data/posentity.txt", "r")
entity = inp.read().splitlines()
inp.close()
for c in entity:
    z = c.split(".")
    elem=z[-1].strip().lower()
    data[elem]=["entity"]

In [ ]:
print(data)

In [ ]:
for elem in data:
    n=0
    allembadding=[]
    with open('./newDir/'+elem.lower()+'.java',encoding = "ISO-8859-1",errors="ignore") as f:
        print(elem)
        lines=f.read()
        code_tokenss = tokenizer.tokenize(lines)
        while(len(code_tokenss)>(n)*510):
            code_tokens=code_tokenss[(n)*510:(n+1)*510]
            tokens = [tokenizer.cls_token] +  code_tokens + [tokenizer.sep_token]
            tokens_ids = tokenizer.convert_tokens_to_ids(tokens)
            context_embeddings = model(torch.tensor(tokens_ids)[None, :])[0]
            embaddings= context_embeddings[0][0].tolist()
            allembadding.append(embaddings)
            n=n+1
        npmatrix=np.matrix(allembadding)
        meanmatrix=npmatrix.mean(0)
        arr=np.array(meanmatrix[0]).flatten()
        data[elem].append(arr.tolist())


In [ ]:
print(data)
with open('posclassEmbadding2.csv', mode='w') as employee_file:
    employee_writer = csv.writer(employee_file, delimiter=';', quotechar='"', quoting=csv.QUOTE_MINIMAL)
    for n in data:
        print(n)
        employee_writer.writerow([n,data[n][0], data[n][1]])